In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

filepath = "GroupExam2024_data_and_documentation/elektronisk-rapportering-ers-2018-fangstmelding-dca-simple.csv"
#Leser filen
df = pd.read_csv(filepath, sep=';', decimal=',')

In [ ]:
#Sile ut informasjon:

df['Bruttotonnasje Kombinert'] = df['Bruttotonnasje 1969'].fillna(df['Bruttotonnasje annen'])

# Dropper un?dvendig informasjon. Tabellen inneholder masse duplikater. som egen kolonne for navn og for kode
# Forskjellige klassifiseringsm?ter. Holder p? FDIR sine.

df.drop(columns = ['Meldingsdato', 
                   'Meldingsklokkeslett', 
                   'Startdato', 
                   'Startklokkeslett', 
                   'Hovedomr?de start', 
                   'Lokasjon start (kode)', 
                   'Stoppdato', 
                   'Stoppklokkeslett', 
                   'Fangst?r', 
                   'Hovedomr?de stopp (kode)', 
                   'Hovedomr?de stopp', 
                   'Lokasjon stopp (kode)', 
                   'Redskap FAO (kode)', 
                   'Redskap FAO', 
                   'Redskap FDIR', 
                   'Hovedart FAO (kode)', 
                   'Hovedart FAO', 
                   'Art FAO (kode)',
                   'Art FAO',
                   'Art - gruppe', 
                   'Lengdegruppe', 
                   'Bredde', 
                   'Fart?ylengde',
                   'Bruttotonnasje 1969',
                   'Bruttotonnasje annen'
    ], inplace=True)

In [ ]:
# Siden Hyse og torsk ofte fiskes samtidig:
filtered_df = df[(['Art - FDIR (kode)'] == 1022) | (df['Art - FDIR (kode)'] == 1027)]
filtered_df

In [ ]:
# Samle data om tokter der meldingsID er lik og starttidspunkt er lik
aggregated_trips = df.groupby(['Melding ID', 'Starttidspunkt']).agg({
    'Rundvekt': 'sum',  # Sum the total catch weight for each trip
    'Art - FDIR': lambda x: ', '.join(x.astype(str).unique()),  # Join unique species names
    # Add other columns as needed, specifying how to aggregate each
}).reset_index()

# Renaming columns for clarity if necessary
aggregated_trips.rename(columns={'Rundvekt': 'TotalFangst', 'Art - FDIR': 'ArtsListe'}, inplace=True)

# Display the aggregated DataFrame
aggregated_trips

In [ ]:
# Droppe nullverdier
df_cleaned = filtered_df.dropna(subset=['Bruttotonnasje Kombinert','Varighet', 'Havdybde start', 'Rundvekt'])

In [ ]:
# Assuming `df` is your DataFrame
# Select features and target
X = df_cleaned[['Bruttotonnasje Kombinert','Varighet', 'Rundvekt']]  # Example features
y = df_cleaned['Havdybde start']  # Target variable


In [ ]:


# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features (for KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# KNN Regressor
knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)
print(f"KNN RMSE: {mean_squared_error(y_test, y_pred_knn, squared=False)}")
print(f'Score: {knn.score(X_test, y_pred_knn)}')

# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print(f"Linear Regression RMSE: {mean_squared_error(y_test, y_pred_lr, squared=False)}")
print(f'Score: {lr.score(X_test, y_pred_knn)}')

Gj?r en standarddeviation av 

In [ ]:
# Assuming df_cleaned is your cleaned DataFrame containing the 'Rundvekt' column
# Step 1: Calculate the mean and standard deviation
mean_catch_weight = df_cleaned['Rundvekt'].mean()
std_dev_catch_weight = df_cleaned['Rundvekt'].std()

# Step 2: Define bins based on standard deviation
lower_bound = mean_catch_weight - std_dev_catch_weight
upper_bound = mean_catch_weight + std_dev_catch_weight

# Step 3: Implement binning
labels = ['Below Average', 'Average', 'Above Average']
df_cleaned['fiske-kategori'] = pd.cut(df_cleaned['Rundvekt'], bins=[-np.inf, lower_bound, upper_bound, np.inf], labels=labels)

print(std_dev_catch_weight)
print(mean_catch_weight)
# Display the first few rows to verify the binning
df_cleaned[['fiske-kategori']].value_counts()


In [ ]:
"""from sklearn.preprocessing import LabelEncoder

# Assuming df is your DataFrame containing the fishing data
# Replace 'round_weight' with the actual column name for round weight in your DataFrame

# Define the bins for categorizing round weight
bins = [-float('inf'), 5000, 10000, float('inf')]
labels = ['bad fishing', 'average fishing', 'good fishing']

# Bin the round weight into categories
df_cleaned['fiske-kategori'] = pd.cut(df_cleaned['Rundvekt'], bins=bins, labels=labels)
df_cleaned
# Now you can use the 'fishing_category' column for classification"""




X = df_cleaned[['Bruttotonnasje Kombinert','Varighet', 'Rundvekt']]  # Example features
y = df_cleaned['Havdybde start']  # Target variable

All fisk:
Resultat med 5 nermeste naboer:
KNN RMSE: 190.3080985365868
Linear Regression RMSE: 200.28353086847204

10 nermeste:
KNN RMSE: 185.96015957933645
Linear Regression RMSE: 200.28353086847204

15 nermeste:
KNN RMSE: 184.5590055924737
Linear Regression RMSE: 200.28353086847204


Filtrert for bare torsk og hyse k=15:
KNN RMSE: 113.03576808665306
Linear Regression RMSE: 120.12048476745328

KNN RMSE: 112.35307942799963
Linear Regression RMSE: 120.12048476745328

Legge inn forloop som itererer gjennom antall naboer og printer score.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

X = df_cleaned[['Bruttotonnasje Kombinert', 'Trekkavstand', 'Havdybde start']]  # Features
y = df_cleaned['fiske-kategori']  # Target variable
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)  # Use transform, not fit_transform

# KNN Classifier
knn_classifier = KNeighborsClassifier(n_neighbors=10)
knn_classifier.fit(X_train_scaled, y_train_encoded)
y_pred_knn_classifier = knn_classifier.predict(X_test_scaled)

# Evaluate the classifier
accuracy = accuracy_score(y_test_encoded, y_pred_knn_classifier)
print(f"KNN Classifier Accuracy: {accuracy}")
print(f'kNN Classifier Score {knn_classifier.score(X_test_scaled, y_test_encoded)}')

### Analyse av score:
Denne koden gir god score. KNN Classifie Det vi m? v?re obs p? her er ?rsak virkning av Trekkavstand.
Dersom man avbryter et fiske tidlig (av andre grunner enn fiske i seg selv) vil dette kanskje registreres. Lite fisk vil bli fisket, og liten avstand blir registrert.  